In [1]:
import kagglehub


path = kagglehub.dataset_download("theredlad/pannuke-dataset-experimental-data")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/pannuke-dataset-experimental-data


In [2]:
import os, re, random, numpy as np, torch, cv2
from PIL import Image
from glob import glob
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision.transforms import functional as TF
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import albumentations as A
from albumentations.pytorch import ToTensorV2

seed=42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.benchmark=True
device=torch.device('cuda' if torch.cuda.is_available() else 'cpu')


root='/kaggle/input/pannuke-dataset-experimental-data'
img_size=512
epochs = 30
bs=4


train_transform = A.Compose([
    A.Resize(height=img_size, width=img_size),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.5),
    A.RandomRotate90(p=0.5),
    A.Affine(translate_percent={"x": 0.05, "y": 0.05}, scale=(0.9, 1.1), rotate=(-15, 15), p=0.5),
    A.ElasticTransform(p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], is_check_shapes=False)

val_transform = A.Compose([
    A.Resize(height=img_size, width=img_size),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], is_check_shapes=False)

test_transform = A.Compose([
    A.Resize(height=img_size, width=img_size),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2(),
], is_check_shapes=False)

In [3]:
from pathlib import Path
import os, shutil, json, numpy as np, torch
from torch.utils.data import Dataset, DataLoader, random_split
from PIL import Image
from torchvision.transforms import functional as TF
from scipy.io import loadmat

def natsort_key(s):
    return [int(t) if t.isdigit() else t.lower() for t in re.split('(\d+)', s)]

def find_pairs_pannuke(split_root):
    """Return (image_files, mask_files) for a given split (train/validate)."""
    img_dir = os.path.join(split_root, "images")
    mask_dir = os.path.join(split_root, "masks")

    exts = ('*.png','*.jpg','*.jpeg','*.tif','*.tiff','*.bmp')
    imgs = []
    masks = []
    for e in exts:
        imgs += glob(os.path.join(img_dir, e))
        masks += glob(os.path.join(mask_dir, e))

    imgs = sorted(imgs, key=natsort_key)
    masks = sorted(masks, key=natsort_key)

    n = min(len(imgs), len(masks))
    return imgs[:n], masks[:n]


train_imgs, train_msks = find_pairs_pannuke(os.path.join(root, "train"))
val_imgs, val_msks = find_pairs_pannuke(os.path.join(root, "validate"))


n_val = len(val_imgs) // 2
va_imgs, te_imgs = val_imgs[:n_val], val_imgs[n_val:]
va_msks, te_msks = val_msks[:n_val], val_msks[n_val:]


all_vals=set()
for p in train_msks + val_msks:
    a=np.array(Image.open(p).convert('L'))
    all_vals.update(np.unique(a).tolist())
vals=sorted(list(all_vals))
if set(vals)=={0,255}: vals=[0,255]
val2idx={v:i for i,v in enumerate(vals)}
num_classes=len(vals)

    
class PanNukeDataset(Dataset):
    def __init__(self, imgs, msks, size, val2idx, training=False):
        self.imgs=imgs; self.msks=msks; self.size=size; self.val2idx=val2idx
        self.training=training
    def __len__(self): return len(self.imgs)
    def _aug(self, x, y):
        if random.random()<0.5:
            x=TF.hflip(x); y=TF.hflip(y)
        if random.random()<0.5:
            x=TF.vflip(x); y=TF.vflip(y)
        if random.random()<0.5:
            ang=random.choice([0,90,180,270])
            x=TF.rotate(x,ang); y=TF.rotate(y,ang,fill=0)
        return x,y
        
    def __getitem__(self, i):
        x=Image.open(self.imgs[i]).convert('RGB')
        y=Image.open(self.msks[i]).convert('L')
        x=x.resize((self.size,self.size), Image.BILINEAR)
        y=y.resize((self.size,self.size), Image.NEAREST)
        if self.training: x,y=self._aug(x,y)
        x=TF.to_tensor(x)
        x=TF.normalize(x,[0.485,0.456,0.406],[0.229,0.224,0.225])
        y=torch.from_numpy(np.vectorize(self.val2idx.get)(np.array(y,dtype=np.uint8))).long()
        return x,y


train_ds=PanNukeDataset(train_imgs,train_msks,img_size,val2idx,training=True)
val_ds=PanNukeDataset(va_imgs,va_msks,img_size,val2idx,training=False)
test_ds=PanNukeDataset(te_imgs,te_msks,img_size,val2idx,training=False)

train_dl=DataLoader(train_ds,batch_size=bs,shuffle=True,num_workers=2,pin_memory=True,drop_last=True)
val_dl=DataLoader(val_ds,batch_size=bs,shuffle=False,num_workers=2,pin_memory=True)
test_dl=DataLoader(test_ds,batch_size=bs,shuffle=False,num_workers=2,pin_memory=True)

print(f"Training samples: {len(train_ds)}")
print(f"Validation samples: {len(val_ds)}")
print(f"Test samples: {len(test_ds)}")
print(f"Train dataloader batches: {len(train_dl)}")
print(f"Validation dataloader batches: {len(val_dl)}")
print(f"Test dataloader batches: {len(test_dl)}")
print(f"Number of classes: {num_classes}")

<>:9: SyntaxWarning: invalid escape sequence '\d'
<>:9: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_24/3290726339.py:9: SyntaxWarning: invalid escape sequence '\d'
  return [int(t) if t.isdigit() else t.lower() for t in re.split('(\d+)', s)]


Training samples: 6716
Validation samples: 592
Test samples: 593
Train dataloader batches: 1679
Validation dataloader batches: 148
Test dataloader batches: 149
Number of classes: 2


In [4]:
class DiceLoss(nn.Module):
    def __init__(self, eps=1e-6): 
        super().__init__(); self.eps=eps
    def forward(self, logits, target):
        if logits.shape[1]==1:
            p=torch.sigmoid(logits)
            t=target.float().unsqueeze(1)
            inter=(p*t).sum(dim=(2,3))
            den=(p+t).sum(dim=(2,3))
            dice=(2*inter+self.eps)/(den+self.eps)
            return 1-dice.mean()
        else:
            p=F.softmax(logits,dim=1)
            t=F.one_hot(target,logits.shape[1]).permute(0,3,1,2).float()
            inter=(p*t).sum(dim=(2,3))
            den=(p+t).sum(dim=(2,3))
            dice=(2*inter+self.eps)/(den+self.eps)
            return 1-dice.mean()

def iou_score(logits, target, eps=1e-6):
    if logits.shape[1]==1:
        p=(torch.sigmoid(logits)>0.5).long()
        t=target.unsqueeze(1)
        inter=((p==1)&(t==1)).sum(dim=(2,3)).float()
        union=((p==1)|(t==1)).sum(dim=(2,3)).float()
        iou=((inter+eps)/(union+eps)).mean().item()
        return iou
    else:
        p=logits.argmax(dim=1)
        ious=[]
        C=logits.shape[1]
        for c in range(C):
            pc=(p==c); tc=(target==c)
            inter=(pc&tc).sum().float()
            union=(pc|tc).sum().float()
            if union==0: continue
            ious.append(((inter+eps)/(union+eps)).item())
        return float(np.mean(ious)) if ious else 0.0

def dice_score(logits, target, eps=1e-6):
    if logits.shape[1]==1:
        p=(torch.sigmoid(logits)>0.5).long()
        t=target.unsqueeze(1)
        inter=((p==1)&(t==1)).sum(dim=(2,3)).float()
        den=(p.sum(dim=(2,3)).float()+t.sum(dim=(2,3)).float())
        return ((2*inter+eps)/(den+eps)).mean().item()
    else:
        p=logits.argmax(dim=1)
        dices=[]
        C=logits.shape[1]
        for c in range(C):
            pc=(p==c); tc=(target==c)
            inter=(pc&tc).sum().float()
            den=pc.sum().float()+tc.sum().float()
            if den==0: continue
            dices.append(((2*inter+eps)/(den+eps)).item())
        return float(np.mean(dices)) if dices else 0.0


In [5]:
def gradient_map(x: torch.Tensor):
    if x.dim() == 2:              
        x = x.unsqueeze(0).unsqueeze(0)
    elif x.dim() == 3:            
        x = x.unsqueeze(1)
    elif x.dim() != 4:
        raise ValueError(f"gradient_map: expected 2D/3D/4D tensor, got {x.dim()}D")

    B, C, H, W = x.shape
    x = x.to(dtype=torch.float32)

    kx = torch.tensor([[1, 0, -1],
                       [2, 0, -2],
                       [1, 0, -1]], dtype=x.dtype, device=x.device).view(1, 1, 3, 3)
    ky = torch.tensor([[ 1,  2,  1],
                       [ 0,  0,  0],
                       [-1, -2, -1]], dtype=x.dtype, device=x.device).view(1, 1, 3, 3)

    kx = kx.repeat(C, 1, 1, 1)  
    ky = ky.repeat(C, 1, 1, 1)  

    gx = F.conv2d(x, kx, padding=1, groups=C)  
    gy = F.conv2d(x, ky, padding=1, groups=C)  
    mag = torch.sqrt(gx * gx + gy * gy + 1e-8)

    if C > 1:
        gx = gx.mean(dim=1, keepdim=True)
        gy = gy.mean(dim=1, keepdim=True)
        mag = mag.mean(dim=1, keepdim=True)

    return gx, gy, mag

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.fft

class FocalFrequencyLoss(nn.Module):
    """
    Optimizes the frequency domain to recover sharp edges (high frequencies).
    """
    def __init__(self, loss_weight=1.0, alpha=1.0, patch_factor=1, ave_spectrum=False, log_matrix=False, batch_matrix=False):
        super(FocalFrequencyLoss, self).__init__()
        self.loss_weight = loss_weight
        self.alpha = alpha
        self.patch_factor = patch_factor
        self.ave_spectrum = ave_spectrum
        self.log_matrix = log_matrix
        self.batch_matrix = batch_matrix

    def tensor2freq(self, x):

        freq = torch.fft.rfft2(x, norm='ortho')
        return torch.stack([freq.real, freq.imag], -1)

    def loss_formulation(self, recon_freq, real_freq, matrix=None):
        if matrix is not None:
            weight_matrix = matrix.detach()
        else:

            matrix_tmp = (recon_freq - real_freq) ** 2
            


            matrix_tmp = matrix_tmp[..., 0] + matrix_tmp[..., 1]

            if self.log_matrix:
                matrix_tmp = torch.log(matrix_tmp + 1.0)



            weight_matrix = torch.clamp(matrix_tmp, min=1e-8, max=float('inf'))
            weight_matrix = weight_matrix ** (self.alpha / 2.0)
            


        weight_matrix = weight_matrix.unsqueeze(-1)

        loss = torch.sum(weight_matrix * (recon_freq - real_freq) ** 2)
        return loss

    def forward(self, pred, target):
        pred = pred.contiguous()
        target = target.contiguous()
        pred_freq = self.tensor2freq(pred)
        target_freq = self.tensor2freq(target)
        return self.loss_weight * self.loss_formulation(pred_freq, target_freq)
        
class MorphologicalLoss(nn.Module):
    """
    Optimizes the structural topology using differentiable erosion/dilation.
    """
    def __init__(self, kernel_size=5):
        super(MorphologicalLoss, self).__init__()
        self.kernel_size = kernel_size
        self.padding = kernel_size // 2

    def erosion(self, x):

        return -F.max_pool2d(-x, kernel_size=self.kernel_size, stride=1, padding=self.padding)

    def dilation(self, x):
        return F.max_pool2d(x, kernel_size=self.kernel_size, stride=1, padding=self.padding)

    def forward(self, pred, target):
        pred_eroded = self.erosion(pred)
        target_eroded = self.erosion(target)
        loss_erosion = F.mse_loss(pred_eroded, target_eroded)
        
        pred_dilated = self.dilation(pred)
        target_dilated = self.dilation(target)
        loss_dilation = F.mse_loss(pred_dilated, target_dilated)
        
        return loss_erosion + loss_dilation

class SpecMorphLoss(nn.Module):
    def __init__(self, lambda_dice=1.0, lambda_focal=0.1, lambda_morph=0.5, num_classes=1):
        super(SpecMorphLoss, self).__init__()
        self.lambda_dice = lambda_dice
        self.lambda_focal = lambda_focal
        self.lambda_morph = lambda_morph
        self.num_classes = num_classes
        

        self.dice = DiceLoss() 
        self.focal_freq = FocalFrequencyLoss()
        self.morph = MorphologicalLoss(kernel_size=5)
        self.ce = nn.CrossEntropyLoss() if num_classes > 1 else nn.BCEWithLogitsLoss()

    def forward(self, logits, target):

        if self.num_classes == 1:
            probs = torch.sigmoid(logits)
            target_float = target.float().unsqueeze(1) if target.dim() == 3 else target.float()
            loss_base = self.lambda_base * (self.dice(probs, target) + self.ce(logits, target_float))

        else:
            probs = F.softmax(logits, dim=1)
            target_one_hot = F.one_hot(target, self.num_classes).permute(0, 3, 1, 2).float()
            loss_base = self.lambda_base * (self.dice(probs, target) + self.ce(logits, target.long()))
            target_float = target_one_hot


        if self.num_classes == 1:
            loss_spec = self.focal_freq(probs, target_float)
        else:
            loss_spec = 0
            for c in range(self.num_classes):
                loss_spec += self.focal_freq(probs[:, c:c+1], target_float[:, c:c+1])
            loss_spec /= self.num_classes


        if self.num_classes == 1:
            loss_morph = self.morph(probs, target_float)
        else:
            loss_morph = 0
            for c in range(self.num_classes):
                loss_morph += self.morph(probs[:, c:c+1], target_float[:, c:c+1])
            loss_morph /= self.num_classes

        return loss_base + (self.lambda_focal * loss_spec) + (self.lambda_morph * loss_morph)

In [7]:
class SEModule(nn.Module):
    def __init__(self, ch, r=16):
        super().__init__()
        self.pool=nn.AdaptiveAvgPool2d(1)
        self.fc=nn.Sequential(nn.Conv2d(ch, ch//r, 1, bias=False), nn.ReLU(inplace=True), nn.Conv2d(ch//r, ch, 1, bias=False), nn.Sigmoid())
    
    def forward(self,x):
        w=self.fc(self.pool(x))
        return x*w

class HINBlock(nn.Module):
    def __init__(self, ch, drop=0.0, se=True):
        super().__init__()
        self.c1=nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.c2=nn.Conv2d(ch, ch, 3, padding=1, bias=False)
        self.bn1=nn.BatchNorm2d(ch)
        self.bn2=nn.BatchNorm2d(ch)
        self.inorm=nn.InstanceNorm2d(ch//2, affine=True)
        self.act=nn.LeakyReLU(0.1, inplace=True)
        self.do=nn.Dropout2d(drop) if drop>0 else nn.Identity()
        self.se=SEModule(ch) if se else nn.Identity()
    
    def forward(self,x):
        c=x.size(1)//2
        a,b=x[:,:c],x[:,c:]
        a=self.inorm(a)
        x=torch.cat([a,b],1)
        y=self.act(self.bn1(self.c1(x)))
        y=self.do(self.act(self.bn2(self.c2(y))))
        y=self.se(y)
        return x+y

class DownBlock(nn.Module):
    def __init__(self, in_ch, out_ch, drop=0.0):
        super().__init__()
        self.down=nn.Conv2d(in_ch,out_ch,3,stride=2,padding=1,bias=False)
        self.bn=nn.BatchNorm2d(out_ch)
        self.act=nn.LeakyReLU(0.1,inplace=True)
        self.h1=HINBlock(out_ch,drop)
        self.h2=HINBlock(out_ch,drop)
    
    def forward(self,x):
        x=self.act(self.bn(self.down(x)))
        x=self.h1(x); x=self.h2(x)
        return x

class DecoderBlock(nn.Module):
    def __init__(self,in_ch,out_ch):
        super().__init__()
        self.conv1=nn.Conv2d(in_ch,in_ch//4,1,bias=False)
        self.bn1=nn.BatchNorm2d(in_ch//4)
        self.deconv=nn.ConvTranspose2d(in_ch//4,in_ch//4,3,stride=2,padding=1,output_padding=1,bias=False)
        self.bn2=nn.BatchNorm2d(in_ch//4)
        self.conv2=nn.Conv2d(in_ch//4,out_ch,1,bias=False)
        self.bn3=nn.BatchNorm2d(out_ch)
        self.act=nn.LeakyReLU(0.1,inplace=True)
        self.se=SEModule(out_ch)
    
    def forward(self,x):
        x=self.act(self.bn1(self.conv1(x)))
        x=self.act(self.bn2(self.deconv(x)))
        x=self.act(self.bn3(self.conv2(x)))
        x=self.se(x)
        return x

class HybridLinkUNet(nn.Module):
    def __init__(self,in_ch=3,base=32,stages=4,classes=1,drop=0.1,learnable_choice=True):
        super().__init__()
        ch=[base,base*2,base*4,base*8][:stages]
        self.stages=stages
        self.enc0=nn.Sequential(nn.Conv2d(in_ch,ch[0],7,stride=2,padding=3,bias=False), nn.BatchNorm2d(ch[0]), nn.LeakyReLU(0.1,inplace=True))
        self.pool=nn.MaxPool2d(3,stride=2,padding=1)
        self.encs=nn.ModuleList([DownBlock(ch[0],ch[1],drop),
                                 DownBlock(ch[1],ch[2],drop),
                                 DownBlock(ch[2],ch[3],drop)][:stages-1])
        dec_out=[ch[i-1] if i>0 else ch[0] for i in range(stages)]
        self.decs=nn.ModuleList([DecoderBlock(ch[i],dec_out[i]) for i in range(stages)])
        self.pre_add=nn.ModuleList([nn.Conv2d(ch[i],ch[i],1,bias=False) for i in range(stages)])
        self.post_add=nn.ModuleList([nn.Conv2d(ch[i], dec_out[i],1,bias=False) for i in range(stages)])
        self.assign=[]
        for i in range(stages):
            self.assign.append('res' if i%2==0 else 'skip')
        self.targets=[]
        for i in range(stages):
            t=i-1 if i>0 else 1 if stages>1 else 0
            self.targets.append(t)
        self.cross_add=nn.ModuleList([nn.Conv2d(ch[i], ch[self.targets[i]],1,bias=False) for i in range(stages)])
        self.gates=nn.Parameter(torch.tensor([5.0 if self.assign[i]=='res' else -5.0 for i in range(stages)])) if learnable_choice else None
        self.head=nn.Sequential(nn.Conv2d(ch[0], ch[0],3,padding=1,bias=False), nn.BatchNorm2d(ch[0]), nn.LeakyReLU(0.1,inplace=True), nn.Conv2d(ch[0], classes if classes>1 else 1, 1))
    
    
    def forward(self,x):
        x0=self.enc0(x)
        f0=self.pool(x0)
        feats=[f0]
        cur=f0
        for b in self.encs:
            cur=b(cur); feats.append(cur)
        feats=feats[:self.stages]
        extras=[None for _ in range(self.stages)]
        resinj=[None for _ in range(self.stages)]
        for i in range(self.stages):
            w=torch.sigmoid(self.gates[i]).view(1,1,1,1) if self.gates is not None else torch.ones(1,1,1,1,device=feats[i].device)
            cw=1-w
            t=self.targets[i]
            add=self.cross_add[i](feats[i])
            if add.shape[-2:]!=feats[t].shape[-2:]:
                add=F.interpolate(add,size=feats[t].shape[-2:],mode='bilinear',align_corners=False)
            add=add*cw
            extras[t]=add if extras[t] is None else extras[t]+add
            r=self.post_add[i](feats[i])*w
            resinj[i]=r
        x=None
        outs=[None for _ in range(self.stages)]
        for i in reversed(range(self.stages)):
            y=feats[i]
            if x is not None: y=y+x
            y=y+self.pre_add[i](feats[i])
            if extras[i] is not None:
                ei=extras[i]
                if ei.shape[-2:]!=y.shape[-2:]:
                    ei=F.interpolate(ei,size=y.shape[-2:],mode='bilinear',align_corners=False)
                y=y+ei
            d=self.decs[i](y)
            ri=resinj[i]
            if ri is not None and ri.shape[-2:]!=d.shape[-2:]:
                ri=F.interpolate(ri,size=d.shape[-2:],mode='bilinear',align_corners=False)
            d=d+ri
            outs[i]=d
            x=d
        out=self.head(outs[0])
        out=F.interpolate(out,scale_factor=2,mode='bilinear',align_corners=False)
        return out


model = HybridLinkUNet(
    in_ch=3,
    base=32,
    stages=4,
    classes=(num_classes if num_classes > 1 else 1),
    drop=0.1,
    learnable_choice=True
).to(device)

learning_rate = 1e-4 
opt = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=1e-4)

sched = torch.optim.lr_scheduler.ReduceLROnPlateau(
    opt, 
    mode='min', 
    factor=0.1, 
    patience=2, 
    threshold=1e-4, 
    min_lr=1e-7
)

scaler = torch.amp.GradScaler(enabled=torch.cuda.is_available())
amp_dtype = torch.bfloat16 if (torch.cuda.is_available() and torch.cuda.is_bf16_supported()) else torch.float16

dice_loss = DiceLoss()

specmorph_loss = SpecMorphLoss(
    lambda_dice=1.0, 
    lambda_focal=5e-5,   
    
    lambda_morph=0.5, 
    num_classes=(num_classes if num_classes > 1 else 1)
).to(device)

In [8]:
from tqdm.notebook import tqdm

def train_epoch():
    model.train()
    lossv = 0.0
    

    loop = tqdm(train_dl, desc="Training", leave=False)
    
    for x, y in loop:
        x = x.to(device, non_blocking=True)
        y = (y.float() if num_classes == 1 else y.long()).to(device, non_blocking=True)
        
        opt.zero_grad(set_to_none=True)
        
        with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available(), dtype=amp_dtype):
            logits = model(x)
            loss = specmorph_loss(logits, y)
            
        if not torch.isfinite(loss):
            print("Warning: Non-finite loss, skipping batch")
            continue
            
        scaler.scale(loss).backward()
        scaler.unscale_(opt)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(opt)
        scaler.update()
        
        lossv += loss.item()
        

        loop.set_postfix(loss=loss.item())
        
    return lossv / max(1, len(train_dl))

In [9]:
@torch.no_grad()
def eval_epoch(dl):
    model.eval()
    tot_loss = 0.0
    miou = 0.0
    mdice = 0.0
    n = 0
    for x, y in dl:
        x = x.to(device, non_blocking=True)
        y = (y.float() if num_classes == 1 else y.long()).to(device, non_blocking=True)
        with torch.amp.autocast(device_type='cuda', enabled=torch.cuda.is_available(), dtype=amp_dtype):
            logits = model(x)
            loss = specmorph_loss(logits, y)
        if torch.isfinite(loss):
            tot_loss += loss.item()
        miou += iou_score(logits, y)
        mdice += dice_score(logits, y)
        n += 1
    return tot_loss / max(1, n), miou / max(1, n), mdice / max(1, n)

In [10]:
best = float('inf'); best_state = None; best_epoch = 0
save_path = '/kaggle/working/pannuke_30_dynamic.pt'

for e in range(1, epochs+1):
    tl = train_epoch()
    vl, vi, vd = eval_epoch(val_dl)
    sched.step(vl)
    lr_now = opt.param_groups[0]['lr']
    if vl < best:
        best = vl; best_epoch = e
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        torch.save({'epoch': e, 'model_state': model.state_dict(), 'optimizer_state': opt.state_dict(), 'val_loss': vl, 'val_iou': vi, 'val_dice': vd, 'num_classes': num_classes}, save_path)
        mark = '*'
    else:
        mark = ''
    print(f"Epoch {e:03d}/{epochs} | train_loss={tl:.4f} | val_loss={vl:.4f} | val_iou={vi:.4f} | val_dice={vd:.4f} | lr={lr_now:.6f} {mark}", flush=True)

if best_state is not None:
    model.load_state_dict(best_state)
    print(f"Loaded best model from epoch {best_epoch} | best_val_loss={best:.4f}", flush=True)

Training:   0%|          | 0/1679 [00:00<?, ?it/s]

Epoch 001/30 | train_loss=174.6561 | val_loss=12.5865 | val_iou=0.7705 | val_dice=0.8609 | lr=0.000100 *


Training:   0%|          | 0/1679 [00:00<?, ?it/s]

Epoch 002/30 | train_loss=13.8889 | val_loss=9.0371 | val_iou=0.7866 | val_dice=0.8722 | lr=0.000100 *


Training:   0%|          | 0/1679 [00:00<?, ?it/s]

...

Epoch 028/30 | train_loss=5.0558 | val_loss=4.9567 | val_iou=0.8320 | val_dice=0.9037 | lr=0.000000 


Training:   0%|          | 0/1679 [00:00<?, ?it/s]

Epoch 029/30 | train_loss=5.0847 | val_loss=4.9409 | val_iou=0.8319 | val_dice=0.9037 | lr=0.000000 *


Training:   0%|          | 0/1679 [00:00<?, ?it/s]

Epoch 030/30 | train_loss=5.0714 | val_loss=4.9286 | val_iou=0.8319 | val_dice=0.9036 | lr=0.000000 *
Loaded best model from epoch 30 | best_val_loss=4.9286


In [11]:
import pandas as pd

bce_loss = nn.BCEWithLogitsLoss()
ce_loss = nn.CrossEntropyLoss()

@torch.no_grad()
def evaluate_test_full():
    model.eval()
    tot_loss = 0.0
    total = 0
    correct = 0
    
    if num_classes == 1:
        tp = 0; fp = 0; fn = 0; tn = 0
        vals = [1]
    else:
        C = num_classes
        tp = torch.zeros(C, dtype=torch.long)
        fp = torch.zeros(C, dtype=torch.long)
        fn = torch.zeros(C, dtype=torch.long)
        tn = torch.zeros(C, dtype=torch.long)
        vals = list(range(C))

    
    for x, y in test_dl:
        x = x.to(device); y = y.to(device)
        logits = model(x)
        if num_classes == 1:
            yt = y.squeeze(1) if (y.dim() == 4 and y.shape[1] == 1) else y
            loss = dice_loss(logits, yt) + 0.5 * bce_loss(logits, yt.float().unsqueeze(1))
            p = (torch.sigmoid(logits) > 0.5).long().squeeze(1)
            t = yt.long()
            total += t.numel()
            correct += (p == t).sum().item()
            tp += ((p == 1) & (t == 1)).sum().item()
            fp += ((p == 1) & (t == 0)).sum().item()
            fn += ((p == 0) & (t == 1)).sum().item()
            tn += ((p == 0) & (t == 0)).sum().item()
        else:
            yt = y.squeeze(1) if (y.dim() == 4 and y.shape[1] == 1) else y
            loss = dice_loss(logits, yt) + 0.5 * ce_loss(logits, yt.long())
            p = logits.argmax(1)
            t = yt.long()
            total += t.numel()
            correct += (p == t).sum().item()
            for c in range(num_classes):
                pc = (p == c)
                tc = (t == c)
                tp[c] += (pc & tc).sum().cpu()
                fp[c] += (pc & (~tc)).sum().cpu()
                fn[c] += ((~pc) & tc).sum().cpu()
                tn[c] += ((~pc) & (~tc)).sum().cpu()
        tot_loss += loss.item()
    avg_loss = tot_loss / len(test_dl)

    
    if num_classes == 1:
        tp_np = tp; fp_np = fp; fn_np = fn; tn_np = tn
        iou = tp_np / (tp_np + fp_np + fn_np + 1e-6)
        dice = 2 * tp_np / (2 * tp_np + fp_np + fn_np + 1e-6)
        precision = tp_np / (tp_np + fp_np + 1e-6)
        recall = tp_np / (tp_np + fn_np + 1e-6)
        specificity = tn_np / (tn_np + fp_np + 1e-6)
        f1_score = 2 * precision * recall / (precision + recall + 1e-6)
        pixel_acc = correct / (total + 1e-6)
        summary = {'loss': avg_loss, 'Accuracy': pixel_acc, 'Precision': precision, 'Recall': recall, 'Specificity': specificity, 'Sensitivity': recall, 'F1-Score': f1_score, 'Dice': dice, 'IoU': iou, 'PixelAcc': pixel_acc, 'num_classes': 1}
        df = pd.DataFrame([{'class_value': 1, 'TP': tp_np, 'FP': fp_np, 'FN': fn_np, 'TN': tn_np, 'Accuracy': pixel_acc, 'Precision': precision, 'Recall': recall, 'Specificity': specificity, 'F1-Score': f1_score, 'Dice': dice, 'IoU': iou}])
    
    else:
        tp_np = tp.numpy(); fp_np = fp.numpy(); fn_np = fn.numpy(); tn_np = tn.numpy()
        iou_c = tp_np / (tp_np + fp_np + fn_np + 1e-6)
        dice_c = 2 * tp_np / (2 * tp_np + fp_np + fn_np + 1e-6)
        precision_c = tp_np / (tp_np + fp_np + 1e-6)
        recall_c = tp_np / (tp_np + fn_np + 1e-6)
        specificity_c = tn_np / (tn_np + fp_np + 1e-6)
        f1_c = 2 * precision_c * recall_c / (precision_c + recall_c + 1e-6)
        valid = (tp_np + fn_np) > 0
        miou = float(iou_c[valid].mean()) if valid.any() else 0.0
        mdice = float(dice_c[valid].mean()) if valid.any() else 0.0
        mprecision = float(precision_c[valid].mean()) if valid.any() else 0.0
        mrecall = float(recall_c[valid].mean()) if valid.any() else 0.0
        mspecificity = float(specificity_c[valid].mean()) if valid.any() else 0.0
        mf1 = float(f1_c[valid].mean()) if valid.any() else 0.0
        pixel_acc = correct / (total + 1e-6)
        summary = {'loss': avg_loss, 'Accuracy': pixel_acc, 'Precision': mprecision, 'Recall': mrecall, 'Specificity': mspecificity, 'Sensitivity': mrecall, 'F1-Score': mf1, 'Dice': mdice, 'IoU': miou, 'PixelAcc': pixel_acc, 'num_classes': num_classes}
        df = pd.DataFrame({'class_value': vals, 'TP': tp_np, 'FP': fp_np, 'FN': fn_np, 'TN': tn_np, 'Accuracy': [pixel_acc] * num_classes, 'Precision': precision_c, 'Recall': recall_c, 'Specificity': specificity_c, 'F1-Score': f1_c, 'Dice': dice_c, 'IoU': iou_c})
    
    os.makedirs('/kaggle/working', exist_ok=True)
    df.to_csv('/kaggle/working/test_per_class_metrics.csv', index=False)
    
    with open('/kaggle/working/test_summary.json', 'w') as f:
        json.dump(summary, f, indent=2)
    
    print(f"Test Loss: {summary['loss']:.6f}")
    print(f"Accuracy: {summary['Accuracy']:.6f}")
    print(f"Precision: {summary['Precision']:.6f}")
    print(f"Recall: {summary['Recall']:.6f}")
    print(f"Specificity: {summary['Specificity']:.6f}")
    print(f"Sensitivity: {summary['Sensitivity']:.6f}")
    print(f"F1-Score: {summary['F1-Score']:.6f}")
    print(f"Dice: {summary['Dice']:.6f}")
    print(f"IoU: {summary['IoU']:.6f}")
    print(f"Pixel Accuracy: {summary['PixelAcc']:.6f}")
    print('\nSaved /kaggle/working/test_per_class_metrics.csv')
    print('Saved /kaggle/working/test_summary.json')
    return summary, df

evaluate_test_full()

Test Loss: 0.226911
Accuracy: 0.948038
Precision: 0.905312
Recall: 0.906233
Specificity: 0.906233
Sensitivity: 0.906233
F1-Score: 0.905771
Dice: 0.905772
IoU: 0.833872
Pixel Accuracy: 0.948038

Saved /kaggle/working/test_per_class_metrics.csv
Saved /kaggle/working/test_summary.json


({'loss': 0.22691065323032789,
  'Accuracy': 0.9480376734098273,
  'Precision': 0.9053122663846576,
  'Recall': 0.9062331929292471,
  'Specificity': 0.9062331929292471,
  'Sensitivity': 0.9062331929292471,
  'F1-Score': 0.9057713486384438,
  'Dice': 0.9057718486376567,
  'IoU': 0.83387229566471,
  'PixelAcc': 0.9480376734098273,
  'num_classes': 2},
    class_value         TP       FP       FN         TN  Accuracy  Precision  \
 0            0  125742668  4002496  4075120   21631108  0.948038   0.969151   
 1            1   21631108  4075120  4002496  125742668  0.948038   0.841473   
 
      Recall  Specificity  F1-Score      Dice       IoU  
 0  0.968609     0.843857  0.968879  0.968880  0.939638  
 1  0.843857     0.968609  0.842663  0.842664  0.728106  )